In [2]:
#Cell 1 - Workspace Setup
#Part A2

from pathlib import Path

#Define project directory
BASE_DIR = Path(r"D:\UTM KL\Course\Y3S1\AI Engineering\Assignment")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

#Create directories
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

#Compatibility aliases
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

#Verify directories
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\raw
Processed folder: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\processed


In [3]:
#Cell 3 - Verify Uploaded Inputs
#Part A2

from pathlib import Path

#Define directories
RAW = RAW_DIR
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BASE = BASE_DIR
OUT = PROCESSED_DIR

#List input files
files = sorted(f for f in RAW.rglob("*") if f.is_file())

print("Input count:", len(files))

for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

#Verify inputs
assert files, "Upload at least one relevant input file before proceeding."

print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
mechanical_science_vol1.pdf bytes: 3166437
mechanical_science_vol2.pdf bytes: 3091979
Upload verified. Select your relevant input(s) for Step A3.


In [4]:
#Part A3 - Inspect Selected Data Sources

from pypdf import PdfReader

#Inspect both PDF files
for f in files:
    reader = PdfReader(str(f))

    print("\nSource:", f.name)
    print("Data type: Unstructured (PDF)")
    print("Total pages:", len(reader.pages))

    #Extract sample text from the first readable page
    for i in range(min(10, len(reader.pages))):
        text = (reader.pages[i].extract_text() or "").strip()

        if text:
            print("Sample page:", i + 1)
            print("Extracted characters:", len(text))
            print("Content sample:", repr(text[:300]))
            break
    else:
        print("No readable text found in the first 10 pages.")


Source: mechanical_science_vol1.pdf
Data type: Unstructured (PDF)
Total pages: 122
Sample page: 1
Extracted characters: 231
Content sample: 'DOE-HDBK-1018/1-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 1 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A.  Approved for public release; distribution is unlimited.'

Source: mechanical_science_vol2.pdf
Data type: Unstructured (PDF)
Total pages: 130
Sample page: 1
Extracted characters: 230
Content sample: 'DOE-HDBK-1018/2-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 2 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A. Approved for public release; distribution is unlimited.'


In [5]:
#Part A3 - Inspect Document Content

from pypdf import PdfReader

#Inspect page 15 of both PDFs
for f in files:
    reader = PdfReader(str(f))
    page = reader.pages[14]
    text = page.extract_text() or ""

    #Display extracted content
    print("\nSource:", f.name)
    print("Page:", 15)
    print("Characters:", len(text))
    print("Content sample:", repr(text[:500]))


Source: mechanical_science_vol1.pdf
Page: 15
Characters: 721
Content sample: 'Diesel Engine Fundamentals DOE-HDBK-1018/1-93 LIST OF FIGURES\nLIST OF FIGURES (Cont.)\nFigure 22 2-Stroke Intake ............................................ 2 6\nFigure 23 2-Stroke Compression ........................................ 2 7\nFigure 24 2-Stroke Fuel Injection ....................................... 2 7\nFigure 25 2-Stroke Power ............................................ 2 7\nFigure 26 Fuel Injector Cutaway ........................................ 3 1\nFigure 27 Fuel Injector Plunger ..'

Source: mechanical_science_vol2.pdf
Page: 15
Characters: 1517
Content sample: 'Valves DOE-HDBK-1018/2-93 LIST OF FIGURES\nLIST OF FIG URES\nFigure 1 Basic Parts of a Valve ......................................2\nFigure 2 Rising Stems ............................................4\nFigure 3 Nonrising Stems ..........................................5\nFigure 4 Gate Valve .......................................

In [6]:
#Part A3 - Inspect Technical Content

#Inspect page 30 from both documents
for f in files:
    reader = PdfReader(str(f))
    text = reader.pages[29].extract_text() or ""

    #Display sample content
    print("\nSource:", f.name)
    print("Page:", 30)
    print("Characters:", len(text))
    print("Content sample:", repr(text[:600]))


Source: mechanical_science_vol1.pdf
Page: 30
Characters: 1610
Content sample: 'DIESEL ENGINES DOE-HDBK-1018/1-93 Diesel Engine Fundamentals\nengine. When the piston travels below the level of the ports, the ports are "opened" and\nfresh air or exhaust gasses are able to enter or leave, depending on the type of port.\nThe ports are then "closed" when the\nFigure 8 Diesel Engine Valve\npiston travels back above the level of\nthe ports. Valves  (refer to figure 8)\nare mechanically opened and closed to\nadmit or exhaust the gasses as needed.\nThe valves are located in the head\ncasting of the engine. The point at\nwhich the valve seals against the head\nis called the valve seat . Most\n'

Source: mechanical_science_vol2.pdf
Page: 30
Characters: 1828
Content sample: 'TYPES OF VALVES DOE-HDBK-1018/2-93 Valves\nOn opening the gate valve, the flow path is enlarged in a highly nonlinear manner with respect\nto  percent  of  opening. This  means  that  flow  rate  does  not  change  evenly  wi

In [7]:
#Cell 4 - Source Selection and Lineage Utilities
#Part B2

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

#Select source file
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )

    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )

    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )

    return matches[0]

#Generate SHA-256 checksum
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

#Save data lineage manifest
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }

    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)

    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("Saved lineage:", target)
    return manifest

#Verify functions
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [8]:
#Cell 4A - Common Lineage Registry
#Part B2

from pathlib import Path

#Initialize lineage registry
lineage_entries = {}

#Register source and output files
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    #Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    #Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )

    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    #Validate processing method
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )

    processing = processing.strip()

    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    #Verify project directories
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    #Record source and output relationship
    key = (str(source), str(output))

    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }

    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

#Verify registry
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [9]:
#Cell 5A1 - PDF Parsing, Cleaning and Chunking
#Part B2

from pypdf import PdfReader
import re

#Select source documents
source_files = [
    select_source(relative_path="mechanical_science_vol1.pdf"),
    select_source(relative_path="mechanical_science_vol2.pdf")
]

#Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]

#Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")

    parts = []
    current = ""

    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()

        if not segment:
            continue

        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment

        else:
            if current:
                parts.append(current)
            current = segment

    if current:
        parts.append(current)

    return parts

#Store results from both documents
documents = {}

for f in source_files:

    #Extract PDF text
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]

    text = "\n".join(content for _, content in pages)

    #Clean extra spaces and tabs
    clean = re.sub(r"[ \t]+", " ", text).strip()

    assert clean, "No usable text found."

    #Compare chunking methods
    fixed = fixed_chunks(clean, size=500)
    structured = paragraph_chunks(clean, size=500)

    #Save results for Cell 5A2
    documents[f.name] = {
        "path": f,
        "pages": len(pages),
        "fixed": fixed,
        "structured": structured
    }

    #Display inspection results
    print("\nSource:", f.name)
    print("Pages:", len(pages))
    print("Extracted characters:", len(text))
    print("Cleaned characters:", len(clean))
    print("Fixed chunks:", len(fixed))
    print("Paragraph-aware chunks:", len(structured))

    print("\nFixed sample:")
    print(repr(fixed[0][:180]))

    print("\nParagraph-aware sample:")
    print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")


Source: mechanical_science_vol1.pdf
Pages: 122
Extracted characters: 169246
Cleaned characters: 166729
Fixed chunks: 334
Paragraph-aware chunks: 341

Fixed sample:
'DOE-HDBK-1018/1-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 1 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A. Appro'

Paragraph-aware sample:
'DOE-HDBK-1018/1-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 1 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A. Appro'

Source: mechanical_science_vol2.pdf
Pages: 130
Extracted characters: 192232
Cleaned characters: 189836
Fixed chunks: 380
Paragraph-aware chunks: 385

Fixed sample:
'DOE-HDBK-1018/2-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 2 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A. Appro'

Paragraph-aware sample:
'DOE-HDBK-1018/2-93\nJANUARY 1993\nDOE FUNDAMENTAL

In [10]:
#Cell 5A2 - Export Prepared Text
#Part B2

import json

#Select chunking method
CHOSEN_METHOD = "paragraph-aware"
approach = "paragraph-aware-char-500"

#Prepare records from both PDFs
records = []

for name, doc in documents.items():
    f = doc["path"]
    chosen = doc["structured"]
    file_hash = source_hash(f)

    #Create records with source information
    for i, chunk in enumerate(chosen, 1):
        records.append({
            "record_id": f"{f.stem}-{i:04d}",
            "text": chunk,
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": file_hash,
            "processing": approach,
            "page": None
        })

#Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"

with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

#Validate exported data
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(exported_records) == len(records), \
    "Exported record count does not match"

#Display results
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 726
Prepared file: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\processed\prepared_text.jsonl
File size: 537464 bytes
Validation: PASS
First record: {'record_id': 'mechanical_science_vol1-0001', 'text': 'DOE-HDBK-1018/1-93\nJANUARY 1993\nDOE FUNDAMENTALS HANDBOOK\nMECHANICAL SCIENCE\nVolume 1 of 2\nU.S. Department of Energy FSC-6910\nWashington, D.C. 20585\nDistribution Statement A. Approved for public release; distribution is unlimited.\nThis document has been reproduced directly from the best available copy.\nAvailable to DOE and DOE contractors from the Office of Scientific and\nTechnical Information. P.O. Box 62, Oak Ridge, TN 37831.\nAvailable to the public from the National Technical Information Serv', 'source': 'mechanical_science_vol1.pdf', 'source_sha256': 'c1e37601164acf95cbaecad93b1db9841ad8d7670bf30a4fdd6ff0a87c6a0070', 'processing': 'paragraph-aware-char-500', 'pag

In [11]:
#Part B3 - Validation and Comparison

#Validate prepared output
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."

print("Output:", prepared_path)
print("Size:", prepared_path.stat().st_size)
print("Validation: PASS")

#Compare chunking methods
for name, doc in documents.items():
    fixed = doc["fixed"]
    structured = doc["structured"]

    print("\nSource:", name)
    print("Fixed chunks:", len(fixed))
    print("Paragraph-aware chunks:", len(structured))

    #Select technical content for comparison
    keyword = "piston" if "vol1" in name else "gate valve"

    fixed_sample = next(
        (c for c in fixed if keyword in c.lower()), fixed[0]
    )
    structured_sample = next(
        (c for c in structured if keyword in c.lower()), structured[0]
    )

    #Display comparison samples
    print("\nFixed-size sample:")
    print(repr(fixed_sample[:300]))

    print("\nParagraph-aware sample:")
    print(repr(structured_sample[:300]))

#Verify total exported records
assert len(exported_records) == 726, "Record count mismatch."

print("\nPart B3 validation completed.")

Output: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\processed\prepared_text.jsonl
Size: 537464
Validation: PASS

Source: mechanical_science_vol1.pdf
Fixed chunks: 334
Paragraph-aware chunks: 341

Fixed-size sample:
're 1 Example of a Large Skid-Mounted, Diesel-Driven Generator ..............2\nFigure 2 Cutaway of a Four-Stroke Supercharged Diesel Engine ..................4\nFigure 3 Cross Section of a V-type Four Stroke Diesel Engine ...................5\nFigure 4 The Cylinder Block ...............................'

Paragraph-aware sample:
'r ..............2\nFigure 2 Cutaway of a Four-Stroke Supercharged Diesel Engine ..................4\nFigure 3 Cross Section of a V-type Four Stroke Diesel Engine ...................5\nFigure 4 The Cylinder Block ...........................................6\nFigure 5 Diesel Engine Wet Cylinder Sleeve ...'

Source: mechanical_science_vol2.pdf
Fixed chunks: 380
Paragraph-aware chunks: 385

Fixed-size sample:
'te Valves ........................

In [12]:
#Part B3 - Technical Content Comparison

#Compare both methods using page 30
for f in source_files:
    reader = PdfReader(str(f))
    text = reader.pages[29].extract_text() or ""
    clean = re.sub(r"[ \t]+", " ", text).strip()

    #Apply both chunking methods
    fixed = fixed_chunks(clean, size=500)
    structured = paragraph_chunks(clean, size=500)

    print("\nSource:", f.name)
    print("Fixed chunks:", len(fixed))
    print("Paragraph-aware chunks:", len(structured))

    #Inspect chunk boundaries
    for method, chunks in [
        ("Fixed-size", fixed),
        ("Paragraph-aware", structured)
    ]:
        print("\nMethod:", method)
        print("Chunk 1 ending:", repr(chunks[0][-120:]))
        if len(chunks) > 1:
            print("Chunk 2 beginning:", repr(chunks[1][:120]))


Source: mechanical_science_vol1.pdf
Fixed chunks: 4
Paragraph-aware chunks: 4

Method: Fixed-size
Chunk 1 ending: 'are mechanically opened and closed to\nadmit or exhaust the gasses as needed.\nThe valves are located in the head\ncasting '
Chunk 2 beginning: 'of the engine. The point at\nwhich the valve seals against the head\nis called the valve seat . Most\nmedium-sized diesels '

Method: Paragraph-aware
Chunk 1 ending: 'are mechanically opened and closed to\nadmit or exhaust the gasses as needed.\nThe valves are located in the head\ncasting '
Chunk 2 beginning: 'of the engine. The point at\nwhich the valve seals against the head\nis called the valve seat . Most\nmedium-sized diesels '

Source: mechanical_science_vol2.pdf
Fixed chunks: 4
Paragraph-aware chunks: 4

Method: Fixed-size
Chunk 1 ending: 'causing disk and seat wear and eventual\nleakage if used to regulate flow. For these reasons, gate valves are not used to'
Chunk 2 beginning: ' regulate or\nthrottle flow.\nA gate valve 

In [13]:
#Part B4.1 - Register Data Lineage

#Register both PDF sources
for f in source_files:
    register_lineage(
        source=f,
        output=prepared_path,
        processing=approach
    )

#Verify registered sources
print("Total lineage entries:", len(lineage_entries))

Registered: mechanical_science_vol1.pdf -> prepared_text.jsonl
Registered: mechanical_science_vol2.pdf -> prepared_text.jsonl
Total lineage entries: 2


In [14]:
#Cell 6A - Generate Common Lineage Manifest
#Part B4.2

#Verify registered lineage
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []

#Generate lineage records
for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    #Validate source and output
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    #Record source metadata
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

#Save common manifest
manifest = save_manifest(lineage_records)

#Verify manifest generation
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\week3_manifest.json


In [15]:
#Cell 6B - Verify Common Lineage Manifest
#Part B4.3

import json

#Locate saved manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"

#Verify manifest exists
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)

#Load manifest
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])

#Validate lineage records
assert saved_records, (
    "No lineage records found in saved manifest."
)

assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

#Verify source and output checksums
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"

    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )

    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

#Display verification results
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\week3_manifest.json


In [16]:
#Cell 7 - Common V1 Pipeline Utilities
#Part C1

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

#Define project paths
BASE_DIR = Path(r"D:\UTM KL\Course\Y3S1\AI Engineering\Assignment")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

#Initialize pipeline registry
PIPELINE_RUNS = {}

#Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

#Select source file or folder
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source

#Execute V1 data pipeline
def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)

    #Ingest selected source files
    if source.is_dir():
        if kind == "documents":
            suffixes = {".pdf", ".txt"}
        elif kind == "images":
            suffixes = {".png", ".jpg", ".jpeg", ".webp"}
        else:
            raise ValueError("Unsupported source folder type.")

        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file() and p.suffix.lower() in suffixes
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    #Inspect source data
    inspection = inspect_fn(inputs, configuration)

    #Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    #Prepare and store data
    outputs, mappings, method = prepare_fn(
        inputs, output_dir, configuration
    )

    outputs = [Path(p) for p in outputs]

    #Validate prepared outputs
    if not outputs:
        raise ValueError("No prepared outputs were produced.")

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    #Record validation results
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    #Record pipeline metadata and lineage
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified - check before sharing"
    }

    #Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record

#Register pipeline for repeatability test
def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }

    return first

#Verify utilities
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [17]:
#Cell 8A - PDF Data Pipeline
#Part C1

import re
import json
from pypdf import PdfReader

#Select PDF source folder
SOURCE_A = resolve_source(".", folder=True)

#Define preparation configuration
CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

#Inspect source documents
def inspect_documents(inputs, cfg):
    results = []

    for src in inputs:
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

        if not text.strip():
            raise ValueError(f"No extractable text: {src.name}")

        results.append({
            "source": src.name,
            "pages": len(reader.pages),
            "characters": len(text)
        })

    return {"documents": results}

#Prepare PDF documents
def prepare_documents(inputs, out, cfg):
    target = out / "prepared_text.jsonl"
    mappings = []
    total_chunks = 0

    with target.open("w", encoding="utf-8") as handle:
        for src in inputs:

            #Extract PDF text
            reader = PdfReader(str(src))
            text = "\n".join(
                page.extract_text() or ""
                for page in reader.pages
            )

            #Clean extracted text
            clean = re.sub(r"[ \t]+", " ", text).strip()

            if not clean:
                raise ValueError(f"Empty document: {src.name}")

            #Apply selected chunking method
            size = int(cfg["chunk_size"])
            method = cfg["method"]

            if method == "fixed":
                chunks = fixed_chunks(clean, size)
            elif method == "paragraph-aware":
                chunks = paragraph_chunks(clean, size)
            else:
                raise ValueError("Invalid chunking method")

            #Export prepared records
            file_hash = sha256_file(src)

            for i, chunk in enumerate(chunks, 1):
                record = {
                    "record_id": f"{src.stem}-{i:04d}",
                    "text": chunk,
                    "source": str(src.relative_to(RAW_DIR)),
                    "source_sha256": file_hash,
                    "processing": f"{method}-char-{size}",
                    "page": None
                }

                handle.write(
                    json.dumps(record, ensure_ascii=False) + "\n"
                )

            total_chunks += len(chunks)

            #Record source-to-output relationship
            mappings.append({
                "source": str(src),
                "output": str(target)
            })

    #Validate prepared data
    assert total_chunks > 0, "No text chunks produced"

    #Compare with Part B output
    previous = PROCESSED_DIR / "prepared_text.jsonl"
    assert previous.is_file(), "Part B output not found"

    assert sha256_file(target) == sha256_file(previous), (
        "Part C output differs from Part B"
    )

    print("Total chunks:", total_chunks)
    print("Part B output comparison: PASS")

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mappings, processing_method

#Execute and register pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)

print("Cell 8A completed successfully.")

Total chunks: 726
Part B output comparison: PASS
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\pipeline_v1\documents\pipeline_manifest.json
Cell 8A completed successfully.


In [18]:
#Cell 9 - Repeatability and Provenance Verification
#Part C1

#Verify registered pipelines
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

#Rerun selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()

    #Verify source fingerprints
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    #Verify prepared output
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    #Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} - source fingerprints, "
        "prepared outputs and mappings match"
    )

#Display verification results
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
Total chunks: 726
Part B output comparison: PASS
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: D:\UTM KL\Course\Y3S1\AI Engineering\Assignment\data\pipeline_v1\documents\pipeline_manifest.json
PASS: documents - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
